# 06 – Tuần 4: PhoBERT S1, S3, thêm seed (W4-01 → W4-03) và XLM-R (W4-04)

**Bắt buộc dùng GPU:** Runtime → Change runtime type → **T4 GPU**.

- Mỗi ô là một task, chạy lần lượt từ trên xuống. Thời gian ghi ở đầu mỗi ô là ước lượng trên T4.
- **Colab bị ngắt giữa chừng thì không sao:** mở lại notebook, chạy ô 0 rồi chạy lại đúng ô đang dở.
  Phần đã có kết quả sẽ tự bỏ qua, chỉ chạy phần còn thiếu.
- Chỉ giữ checkpoint của S1 (S3 cần để học tiếp), khoảng 0,5 GB (PhoBERT) và 1,1 GB (XLM-R) trên Drive chung.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/NLI'
!git clone -q https://github.com/quanuiter/NLI.git /content/NLI 2>/dev/null || git -C /content/NLI pull -q
%cd /content/NLI
!pip install -q -r requirements.txt
import os
for sub in ['data/processed', 'tools', 'results', 'checkpoints']:
    target = os.path.join(DRIVE_DIR, sub); os.makedirs(target, exist_ok=True)
    if os.path.islink(sub): continue
    if os.path.isdir(sub):
        !cp -rn {sub}/. {target}/ && rm -rf {sub}
    os.symlink(target, sub)

In [ ]:
import os, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU, bật T4 GPU rồi chạy lại từ ô 0')
need = [f'data/processed/{f}_seg.jsonl' for f in
        ['xnli_train_100k', 'xnli_dev', 'xnli_test', 'vianli_train', 'vianli_dev', 'vianli_test', 'vinli_test']]
missing = [f for f in need if not os.path.exists(f)]
print('Thiếu file tách từ:', missing if missing else 'không, đủ cả 7 file')

## W4-01: PhoBERT kịch bản S1 (khoảng 30–40 phút)
Học trên XNLI 100.000 cặp, cấu hình A (lr 2e-5, batch 32, 3 epoch), đánh giá XNLI test, ViANLI test, ViNLI test.
Chọn A vì trên dev S2 hai cấu hình ngang nhau (0,444 và 0,448) mà B (7 epoch) trên 100.000 cặp sẽ mất gần 1,5 tiếng.

In [ ]:
!python scripts/run_transformer.py --model M4 --scenario S1 --config A

## W4-02: PhoBERT kịch bản S3 (khoảng 7 phút)
Bắt đầu từ checkpoint S1 (`checkpoints/M4_S1_seed42`), học tiếp trên ViANLI train với cấu hình B (lr 1e-5, nhỏ hơn S1).

In [ ]:
!python scripts/run_transformer.py --model M4 --scenario S3 --config B

## W4-03: PhoBERT thêm seed 123 và 2024 cho S2, S3 (khoảng 30 phút)
S3 của mọi seed đều bắt đầu từ cùng checkpoint S1 seed 42; seed chỉ đổi thứ tự dữ liệu và khởi tạo lớp phân loại khi học tiếp.

In [ ]:
!python scripts/run_transformer.py --model M4 --scenario S2 S3 --config B --seeds 123 2024

## W4-04: XLM-R (M5) kịch bản S1, S2, S3 (khoảng 1 tiếng)
Cùng cấu hình với PhoBERT để so sánh công bằng: S1 cấu hình A, S2 và S3 cấu hình B. XLM-R dùng văn bản thô, không cần tách từ.
Có thể chạy từng dòng nếu sợ bị ngắt giữa chừng.

In [ ]:
!python scripts/run_transformer.py --model M5 --scenario S1 --config A
!python scripts/run_transformer.py --model M5 --scenario S2 S3 --config B

## Kết quả

In [ ]:
from src.evaluate import summarize
t = summarize()
t = t[t['model'].isin(['M2', 'M3', 'M4', 'M5'])]
t[['model', 'scenario', 'test_set', 'accuracy_mean', 'accuracy_std', 'macro_f1_mean', 'macro_f1_std', 'macro_f1_count']].round(3)

Xong thì nhắn Claude "xong tuần 4" để đồng bộ kết quả từ Drive lên GitHub và kiểm tra.